In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/artifacts_final.zip -d /content/dataset/

In [ ]:
!pip install torch torchvision transformers pillow xgboost scikit-learn dask[complete] dask-ml

In [ ]:
import os
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTModel
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import joblib

# Paths to your data directories
data_dir = '/content/dataset/artifact_final'
artifacts_dir = os.path.join(data_dir, 'artifacts')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transformation for input images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Normalization parameters for ViT
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# Custom Dataset class
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Parameters
batch_size = 32

# List of artifact directories
artifact_list = sorted(os.listdir(artifacts_dir))

# Pre-load all artifact images
artifact_images_dict = {}
for artifact_name in artifact_list:
    artifact_folder = os.path.join(artifacts_dir, artifact_name)
    images = [os.path.join(artifact_folder, img) for img in os.listdir(artifact_folder)]
    artifact_images_dict[artifact_name] = images

# Load pre-trained ViT model for feature extraction (load only once)
print("Loading ViT model...")
vit_model = ViTModel.from_pretrained('google/vit-large-patch16-224-in21k')
vit_model.to(device)
vit_model.eval()  # Set model to evaluation mode

xgb_params = {
    'max_depth': 5,
    'n_estimators': 200,
    'learning_rate': 0.05,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'logloss'
}


# Function to extract embeddings
def extract_embeddings(data_loader):
    embeddings = []
    labels_list = []
    with torch.no_grad():
        for images_batch, labels_batch in data_loader:
            images_batch = images_batch.to(device)
            outputs = vit_model(images_batch)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu().numpy())
            labels_list.extend(labels_batch.numpy())
    embeddings = np.vstack(embeddings)
    labels_list = np.array(labels_list)
    return embeddings, labels_list

# Loop over each artifact
for artifact_num in range(len(artifact_list)):
    artifact_name = artifact_list[artifact_num]
    print(artifact_num, f"\nProcessing artifact: {artifact_name}")

    # Get images with the artifact (positive samples)
    artifact_images = artifact_images_dict[artifact_name]
    num_artifact_images = len(artifact_images)

    # Get images from other artifacts (negative samples)
    other_artifact_images = []
    for other_name, images in artifact_images_dict.items():
        if other_name != artifact_name:
            other_artifact_images.extend(images)

    # Ensure there are enough images
    if num_artifact_images > len(other_artifact_images):
        raise ValueError(f"Not enough images in other artifacts for artifact '{artifact_name}'.")

    # Randomly select negative samples to balance the dataset
    negative_images = random.sample(other_artifact_images, num_artifact_images)

    # Combine and label the data
    images = artifact_images + negative_images
    labels = [1] * num_artifact_images + [0] * num_artifact_images

    # Shuffle the dataset
    combined = list(zip(images, labels))
    random.shuffle(combined)
    images[:], labels[:] = zip(*combined)

    # Split into training and validation sets (80% train, 20% validation)
    train_images, val_images, train_labels, val_labels = train_test_split(
        images, labels, test_size=0.2, random_state=42, stratify=labels)

    # Create datasets
    train_dataset = ImageDataset(train_images, train_labels, transform=transform)
    val_dataset = ImageDataset(val_images, val_labels, transform=transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Extract embeddings for training and validation sets
    print("Extracting embeddings for training set...")
    train_embeddings, train_labels_array = extract_embeddings(train_loader)

    print("Extracting embeddings for validation set...")
    val_embeddings, val_labels_array = extract_embeddings(val_loader)

    # Train XGBoost classifier
    print("Training XGBoost classifier...")
    xgb_classifier = XGBClassifier(**xgb_params)
    xgb_classifier.fit(train_embeddings, train_labels_array)

    # Evaluate on validation set
    val_predictions = xgb_classifier.predict(val_embeddings)
    val_accuracy = accuracy_score(val_labels_array, val_predictions)
    print(f'Artifact: {artifact_name}, Validation Accuracy: {val_accuracy * 100:.2f}%')

    # Save the trained XGBoost model in binary format
    model_save_path = f'/content/drive/MyDrive/xgb_models/xgb_model_{artifact_name}.pth'  # Using .pth extension
    joblib.dump(xgb_classifier, model_save_path)
    print(f"Model saved to {model_save_path}")


Loading ViT model...


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]


Processing artifact: excessive_sharpness
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training XGBoost classifier...
Artifact: excessive_sharpness, Validation Accuracy: 88.89%
Model saved to /content/drive/MyDrive/xgb_models/xgb_model_excessive_sharpness.pth

Processing artifact: incorrect_perspective
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training XGBoost classifier...
Artifact: incorrect_perspective, Validation Accuracy: 88.95%
Model saved to /content/drive/MyDrive/xgb_models/xgb_model_incorrect_perspective.pth

Processing artifact: resolution_inconsistencies
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training XGBoost classifier...
Artifact: resolution_inconsistencies, Validation Accuracy: 90.62%
Model saved to /content/drive/MyDrive/xgb_models/xgb_model_resolution_inconsistencies.pth

Processing artifact: unnaturally_glossy_surfaces
Extracting embedding